# Qwen2.5-VL-7B Server for HOT OSM Classification

Serves `Qwen/Qwen2.5-VL-7B-Instruct-AWQ` as an OpenAI-compatible endpoint via vLLM, tunneled through ngrok so `run_api.py` on your laptop can hit it.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free tier is fine).
2. Sign up at https://ngrok.com (free), copy your authtoken from the dashboard.
3. Paste the token into Cell 3 below.

**Runtime:** ~3 min to install + load model, then stays up until Colab times out (~90 min idle on free tier).

## Cell 1 - Verify GPU

In [ ]:
!nvidia-smi

## Cell 2 - Install vLLM + ngrok

In [ ]:
!pip install -q --upgrade vllm pyngrok

## Cell 3 - Start ngrok tunnel

Paste your ngrok authtoken below. You'll get a public URL - **copy it**, you'll paste it into your local `.env` as `COLAB_BASE_URL`.

In [ ]:
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN

# kill any prior tunnels from previous runs of this cell
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url.replace("http://", "https://")
print("\n" + "=" * 60)
print("PUBLIC URL (add to .env as COLAB_BASE_URL):")
print(f"  {public_url}/v1")
print("=" * 60 + "\n")

## Cell 4 - Launch vLLM server

Uses the 4-bit AWQ quant so it fits comfortably on a free T4 (15GB). This cell runs forever - while it's streaming logs, your local `run_api.py` can hit the tunnel.

**To stop:** Runtime → Interrupt execution.

In [ ]:
!python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-VL-7B-Instruct-AWQ \
    --quantization awq \
    --dtype float16 \
    --port 8000 \
    --host 0.0.0.0 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.90